In [ ]:
# --- Cell 1: Setup & Parameters ---
import numpy as np
from scipy.interpolate import BSpline
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings

warnings.filterwarnings('ignore')

SPACE_X, SPACE_Y, SPACE_Z = 1000.0, 1000.0, 120.0

START = np.array([100.0, 100.0, 50.0])
TARGET = np.array([850.0, 900.0, 40.0])

OBSTACLES = [
    [200, 300, 150, 250, 0, 100],
    [400, 600, 300, 450, 0, 70],
    [700, 800, 200, 300, 0, 90],
    [100, 400, 600, 700, 0, 50],
    [500, 650, 650, 800, 0, 85],
    [750, 900, 750, 850, 0, 60],
    [300, 450, 800, 950, 0, 75]
]

OBSTACLES = [np.array(o, dtype=float) for o in OBSTACLES]

N_DISC = 50
KMAX      = 10_000.0
H_MIN     = 10.0
H_MAX     = 100.0
H_MID     = (H_MAX + H_MIN) / 2.0
ALPHA_MAX = np.pi / 4
BETA_MAX  = np.pi / 3
LAMBDA = MU = PHI = GAMMA = 1.0

P_MIN = np.array([0., 0., H_MIN, 0., 0., H_MIN, 0., 0., H_MIN])
P_MAX = np.array([SPACE_X, SPACE_Y, H_MAX, SPACE_X, SPACE_Y, H_MAX, SPACE_X, SPACE_Y, H_MAX])
V_MAX = (P_MAX - P_MIN) * 0.2

N_PARTICLES = 50
T_MAX       = 100
W_MAX, W_MIN = 0.9, 0.4
C_MAX, C_MIN = 2.5, 0.5
A1, A2   = 0.1, 0.3
B1, B2   = 0.4, 0.8
FRAC_D1  = 0.20
FRAC_D2  = 0.50
F_THRESH = KMAX * np.linalg.norm(TARGET - START)
A_ESC    = np.sqrt(1000.0 / N_PARTICLES)

In [14]:
# --- 2: Physics Engine & Objective Function ---
def generate_path(ctrl_pts: np.ndarray, degree: int = 3) -> np.ndarray:
    m = len(ctrl_pts)
    k = min(degree, m - 1)
    n_interior = m - k - 1
    interior   = np.linspace(0.0, 1.0, n_interior + 2)[1:-1] if n_interior > 0 else []
    knots      = np.concatenate([[0.0] * (k + 1), interior, [1.0] * (k + 1)])
    u          = np.linspace(0.0, 1.0, N_DISC)
    return BSpline(knots, ctrl_pts, k)(u)

def segment_intersects_aabb(p1, p2, obs) -> bool:
    box_min = np.array([obs[0], obs[2], obs[4]])
    box_max = np.array([obs[1], obs[3], obs[5]])
    d  = p2 - p1
    t0, t1 = 0.0, 1.0
    for i in range(3):
        if abs(d[i]) < 1e-9:
            if p1[i] < box_min[i] or p1[i] > box_max[i]:
                return False
        else:
            ta = (box_min[i] - p1[i]) / d[i]
            tb = (box_max[i] - p1[i]) / d[i]
            if ta > tb: ta, tb = tb, ta
            t0 = max(t0, ta)
            t1 = min(t1, tb)
            if t0 > t1: return False
    return True

def count_collisions(path: np.ndarray) -> int:
    c = 0
    for i in range(len(path) - 1):
        for obs in OBSTACLES:
            if segment_intersects_aabb(path[i], path[i + 1], obs):
                c += 1
                break
    return c

def fitness(particle: np.ndarray):
    P1   = particle[0:3]
    P2   = particle[3:6]
    P3   = particle[6:9]
    ctrl = np.vstack([START, P1, P2, P3, TARGET])
    path = generate_path(ctrl)

    deltas  = np.diff(path, axis=0)
    seg_len = np.linalg.norm(deltas, axis=1)
    fL      = seg_len.sum()
    fS      = count_collisions(path) * KMAX * fL

    z  = path[:, 2]
    hi = np.zeros(len(path))
    hi[(z < H_MIN) | (z >= H_MAX)]                 = KMAX
    hi[(z >= H_MIN) & (z < H_MID)] = (H_MID - z[(z >= H_MIN) & (z <  H_MID)]) ** 2
    hi[(z >= H_MID) & (z < H_MAX)] = (z[(z >= H_MID) & (z <  H_MAX)] - H_MID) ** 2
    fH = hi.sum()

    dx = deltas[:, 0]; dy = deltas[:, 1]; dz = deltas[:, 2]
    alpha = np.abs(np.arctan2(dz, np.sqrt(dx**2 + dy**2) + 1e-9))
    beta  = np.abs(np.arctan2(dy, dx + 1e-9))
    fA = (np.where(alpha > ALPHA_MAX, KMAX, alpha).sum() +
          np.where(beta  > BETA_MAX,  KMAX, beta ).sum())

    return LAMBDA*fL + MU*fS + PHI*fH + GAMMA*fA, path

In [15]:
# --- 3: MSIPSO Optimizer ---
def adaptive_params(t):
    exp_t  = np.exp(-t / T_MAX)
    exp_1  = np.exp(-1.0)
    exp_dt = np.exp(-1.0 / T_MAX)
    w  = W_MIN + (W_MAX - W_MIN) * (exp_t - exp_1) / (exp_dt - exp_1)
    ln = np.log(1.0 + (np.e - 1.0) * t / T_MAX)
    c1 = C_MAX - (C_MAX - C_MIN) * ln
    c2 = C_MIN + (C_MAX - C_MIN) * ln
    return float(w), float(c1), float(c2)

def run_msipso(seed=42, verbose=True):
    rng      = np.random.default_rng(seed)
    pos      = rng.uniform(P_MIN, P_MAX, (N_PARTICLES, 9))
    vel      = rng.uniform(-V_MAX, V_MAX, (N_PARTICLES, 9))
    fit_vals = np.array([fitness(pos[i])[0] for i in range(N_PARTICLES)])
    pbest_pos = pos.copy()
    pbest_fit = fit_vals.copy()
    gbest_idx = int(np.argmin(pbest_fit))
    gbest_pos = pbest_pos[gbest_idx].copy()
    gbest_fit = float(pbest_fit[gbest_idx])
    
    history   = []
    swarm_p2_history = []
    swarm_iterations = []

    for t in range(1, T_MAX + 1):
        w, c1, c2 = adaptive_params(t)
        deadlock   = gbest_fit > F_THRESH

        if deadlock:
            r3  = rng.random((N_PARTICLES, 9))
            vel = A_ESC - 2.0 * A_ESC * r3
            pos = np.clip(pos + vel, P_MIN, P_MAX)
        else:
            r1  = rng.random((N_PARTICLES, 9))
            r2  = rng.random((N_PARTICLES, 9))
            vel = np.clip(w*vel + c1*r1*(pbest_pos-pos) + c2*r2*(gbest_pos-pos), -V_MAX, V_MAX)

            sorted_idx = np.argsort(fit_vals)
            n_d1   = max(1, int(FRAC_D1 * N_PARTICLES))
            n_d2   = max(1, int(FRAC_D2 * N_PARTICLES))
            idx_d1 = sorted_idx[:n_d1]
            idx_d2 = sorted_idx[n_d1: n_d1 + n_d2]
            idx_d3 = sorted_idx[n_d1 + n_d2:]

            r_d1        = rng.random((len(idx_d1), 9))
            pos[idx_d1] = pos[idx_d1] + (A1 + (A2-A1)*r_d1) * vel[idx_d1]
            pos[idx_d2] = pos[idx_d2] + (B1 + (B2-B1)*t/T_MAX) * vel[idx_d2]
            r4          = rng.random((len(idx_d3), 9))
            pos[idx_d3] = P_MIN + r4 * (P_MAX - P_MIN)
            pos = np.clip(pos, P_MIN, P_MAX)

        for i in range(N_PARTICLES):
            f_i, _ = fitness(pos[i])
            fit_vals[i] = f_i
            if f_i < pbest_fit[i]:
                pbest_fit[i] = f_i
                pbest_pos[i] = pos[i].copy()

        best_i = int(np.argmin(pbest_fit))
        if pbest_fit[best_i] < gbest_fit:
            gbest_fit = float(pbest_fit[best_i])
            gbest_pos = pbest_pos[best_i].copy()

        history.append(gbest_fit)
        
        if t == 1 or t % 5 == 0:
            swarm_p2_history.append(pos[:, 3:6].copy())
            swarm_iterations.extend([t] * N_PARTICLES)

        if verbose and (t == 1 or t % 10 == 0):
            status = "DEADLOCK" if deadlock else "optimising"
            print(f"  Iter {t:3d}/{T_MAX}  [{status}]  fitness={gbest_fit:12.4f}  w={w:.3f}  c1={c1:.3f}  c2={c2:.3f}")

    _, best_path = fitness(gbest_pos)
    return gbest_pos, gbest_fit, best_path, history, np.vstack(swarm_p2_history), np.array(swarm_iterations)

In [ ]:
# --- 4: Plotly Visualization Functions ---
def plot_results_plotly(best_path, history, gbest_pos):
    P1  = gbest_pos[0:3]
    P2  = gbest_pos[3:6]
    P3  = gbest_pos[6:9]
    cps = np.vstack([START, P1, P2, P3, TARGET])

    fig = make_subplots(
        rows=3, cols=2,
        column_widths=[0.65, 0.35],
        specs=[[{'type': 'scene', 'rowspan': 3}, {'type': 'xy'}],
               [None,                            {'type': 'xy'}],
               [None,                            {'type': 'xy'}]],
        subplot_titles=('Interactive 3D UAV Path', 'Top-Down View (XY)', 'Side Elevation View (XZ)', 'Convergence Curve'),
        vertical_spacing=0.08,
        horizontal_spacing=0.05
    )

    for obs in OBSTACLES:
        x0, x1, y0, y1, z0, z1 = obs
        x = [x0, x0, x1, x1, x0, x0, x1, x1]
        y = [y0, y1, y1, y0, y0, y1, y1, y0]
        z = [z0, z0, z0, z0, z1, z1, z1, z1]
        
        fig.add_trace(go.Mesh3d(x=x, y=y, z=z, alphahull=0, color='#56B4E9', opacity=0.6, hoverinfo='skip'), row=1, col=1)
        fig.add_trace(go.Scatter(x=[x0, x1, x1, x0, x0], y=[y0, y0, y1, y1, y0], fill='toself', fillcolor='rgba(86, 180, 233, 0.4)', line=dict(color='#56B4E9', width=1), showlegend=False, hoverinfo='skip'), row=1, col=2)
        fig.add_trace(go.Scatter(x=[x0, x1, x1, x0, x0], y=[z0, z0, z1, z1, z0], fill='toself', fillcolor='rgba(86, 180, 233, 0.4)', line=dict(color='#56B4E9', width=1), showlegend=False, hoverinfo='skip'), row=2, col=2)

    fig.add_trace(go.Scatter(x=[0, SPACE_X, SPACE_X, 0, 0], y=[H_MIN, H_MIN, H_MAX, H_MAX, H_MIN], fill='toself', fillcolor='rgba(0, 158, 115, 0.08)', line=dict(color='rgba(255,255,255,0)', width=0), showlegend=False, hoverinfo='skip'), row=2, col=2)
    fig.add_trace(go.Scatter(x=[0, SPACE_X], y=[H_MIN, H_MIN], mode='lines', line=dict(color="#009E73", dash="dash", width=1), showlegend=False, hoverinfo='skip'), row=2, col=2)
    fig.add_trace(go.Scatter(x=[0, SPACE_X], y=[H_MAX, H_MAX], mode='lines', line=dict(color="#009E73", dash="dash", width=1), showlegend=False, hoverinfo='skip'), row=2, col=2)
    fig.add_trace(go.Scatter(x=[0, SPACE_X], y=[H_MID, H_MID], mode='lines', line=dict(color="#009E73", dash="dot", width=1), showlegend=False, hoverinfo='skip'), row=2, col=2)

    fig.add_trace(go.Scatter3d(x=best_path[:, 0], y=best_path[:, 1], z=best_path[:, 2], mode='lines', line=dict(color='#D55E00', width=6), name='UAV Path'), row=1, col=1)
    fig.add_trace(go.Scatter3d(x=cps[:, 0], y=cps[:, 1], z=cps[:, 2], mode='lines+markers', line=dict(color='#e74c3c', dash='dash', width=2), marker=dict(size=5, color='#e74c3c'), name='Control Polygon'), row=1, col=1)
    fig.add_trace(go.Scatter3d(x=[START[0]], y=[START[1]], z=[START[2]], mode='markers', marker=dict(color='#009E73', size=6), name='Start'), row=1, col=1)
    fig.add_trace(go.Scatter3d(x=[TARGET[0]], y=[TARGET[1]], z=[TARGET[2]], mode='markers', marker=dict(color='#CC79A7', symbol='diamond', size=6), name='Target'), row=1, col=1)

    fig.add_trace(go.Scatter(x=best_path[:, 0], y=best_path[:, 1], mode='lines', line=dict(color='#D55E00', width=3), showlegend=False), row=1, col=2)
    fig.add_trace(go.Scatter(x=cps[:, 0], y=cps[:, 1], mode='lines+markers', line=dict(color='#e74c3c', dash='dash', width=1), marker=dict(size=6, color='#e74c3c'), showlegend=False), row=1, col=2)
    fig.add_trace(go.Scatter(x=[START[0]], y=[START[1]], mode='markers', marker=dict(color='#009E73', size=10), showlegend=False), row=1, col=2)
    fig.add_trace(go.Scatter(x=[TARGET[0]], y=[TARGET[1]], mode='markers', marker=dict(color='#CC79A7', symbol='diamond', size=10), showlegend=False), row=1, col=2)

    fig.add_trace(go.Scatter(x=best_path[:, 0], y=best_path[:, 2], mode='lines', line=dict(color='#D55E00', width=3), showlegend=False), row=2, col=2)
    fig.add_trace(go.Scatter(x=cps[:, 0], y=cps[:, 2], mode='lines+markers', line=dict(color='#e74c3c', dash='dash', width=1), marker=dict(size=6, color='#e74c3c'), showlegend=False), row=2, col=2)
    fig.add_trace(go.Scatter(x=[START[0]], y=[START[2]], mode='markers', marker=dict(color='#009E73', size=10), showlegend=False), row=2, col=2)
    fig.add_trace(go.Scatter(x=[TARGET[0]], y=[TARGET[2]], mode='markers', marker=dict(color='#CC79A7', symbol='diamond', size=10), showlegend=False), row=2, col=2)

    iters = list(range(1, len(history) + 1))
    fig.add_trace(go.Scatter(x=iters, y=history, mode='lines', line=dict(color='#D55E00', width=2), name='Fitness'), row=3, col=2)
    fig.add_trace(go.Scatter(x=[1, len(history)], y=[F_THRESH, F_THRESH], mode='lines', line=dict(color='#999999', dash='dash'), name='F_max (Deadlock)'), row=3, col=2)

    fig.update_layout(
        template='plotly_white', paper_bgcolor='#ffffff', plot_bgcolor='#ffffff', height=950, margin=dict(l=20, r=20, t=60, b=20),
        scene=dict(xaxis=dict(range=[0, SPACE_X], title='X (m)', backgroundcolor='#ffffff', gridcolor='#e5e5e5'), yaxis=dict(range=[0, SPACE_Y], title='Y (m)', backgroundcolor='#ffffff', gridcolor='#e5e5e5'), zaxis=dict(range=[0, SPACE_Z], title='Z (m)', backgroundcolor='#ffffff', gridcolor='#e5e5e5'), aspectmode='manual', aspectratio=dict(x=1, y=1, z=0.4), camera=dict(eye=dict(x=-1.5, y=-1.5, z=0.5))),
        legend=dict(x=0.01, y=0.99, bgcolor='rgba(255, 255, 255, 0.8)', font=dict(color='black'))
    )

    fig.update_xaxes(range=[0, SPACE_X], title_text="X (m)", gridcolor='#e5e5e5', row=1, col=2)
    fig.update_yaxes(range=[0, SPACE_Y], title_text="Y (m)", gridcolor='#e5e5e5', row=1, col=2)
    fig.update_xaxes(range=[0, SPACE_X], title_text="X (m)", gridcolor='#e5e5e5', row=2, col=2)
    fig.update_yaxes(range=[0, SPACE_Z], title_text="Z (Alt m)", gridcolor='#e5e5e5', row=2, col=2)
    fig.update_xaxes(title_text="Iteration", gridcolor='#e5e5e5', row=3, col=2)
    fig.update_yaxes(title_text="Best Fitness", gridcolor='#e5e5e5', type='log', row=3, col=2)
    
    fig.show()

def plot_combined_heatmaps(swarm_p2_history, swarm_iterations, gbest_pos):
    P1_opt = gbest_pos[0:3]
    P3_opt = gbest_pos[6:9]

    frozen_fitness_swarm = []
    for p2 in swarm_p2_history:
        temp_particle = np.concatenate([P1_opt, p2, P3_opt])
        fit_val, _ = fitness(temp_particle)
        frozen_fitness_swarm.append(fit_val)
    frozen_fitness_swarm = np.array(frozen_fitness_swarm)
    log_fitness_swarm = np.log10(frozen_fitness_swarm + 1)

    x_vals = np.linspace(0, SPACE_X, 30)
    y_vals = np.linspace(0, SPACE_Y, 30)
    z_vals = np.linspace(H_MIN, H_MAX, 10)

    X, Y, Z = np.meshgrid(x_vals, y_vals, z_vals, indexing='ij')
    grid_points = np.vstack([X.ravel(), Y.ravel(), Z.ravel()]).T

    frozen_fitness_grid = []
    for p2 in grid_points:
        temp_particle = np.concatenate([P1_opt, p2, P3_opt])
        fit_val, _ = fitness(temp_particle)
        frozen_fitness_grid.append(fit_val)
    frozen_fitness_grid = np.array(frozen_fitness_grid)
    log_fitness_grid = np.log10(frozen_fitness_grid + 1)

    cmin_val = min(log_fitness_swarm.min(), log_fitness_grid.min())
    cmax_val = max(log_fitness_swarm.max(), log_fitness_grid.max())

    fig = make_subplots(
        rows=1, cols=2,
        specs=[[{'type': 'scene'}, {'type': 'scene'}]],
        subplot_titles=('Swarm Tracker Heatmap', 'Full City Voxel Heatmap'),
        horizontal_spacing=0.05
    )

    for obs in OBSTACLES:
        x0, x1, y0, y1, z0, z1 = obs
        x = [x0, x0, x1, x1, x0, x0, x1, x1]
        y = [y0, y1, y1, y0, y0, y1, y1, y0]
        z = [z0, z0, z0, z0, z1, z1, z1, z1]
        fig.add_trace(go.Mesh3d(x=x, y=y, z=z, alphahull=0, color='#56B4E9', opacity=0.15, hoverinfo='skip'), row=1, col=1)
        fig.add_trace(go.Mesh3d(x=x, y=y, z=z, alphahull=0, color='#56B4E9', opacity=0.15, hoverinfo='skip'), row=1, col=2)

    fig.add_trace(go.Scatter3d(
        x=swarm_p2_history[:, 0], y=swarm_p2_history[:, 1], z=swarm_p2_history[:, 2], mode='markers',
        marker=dict(size=4, color=log_fitness_swarm, colorscale='RdYlGn_r', cmin=cmin_val, cmax=cmax_val, showscale=False, opacity=0.8),
        text=[f"Iter: {it}<br>Cost: {f:,.0f}" for it, f in zip(swarm_iterations, frozen_fitness_swarm)], hoverinfo='text', name='P2 Evaluations'
    ), row=1, col=1)
    fig.add_trace(go.Scatter3d(x=[gbest_pos[3]], y=[gbest_pos[4]], z=[gbest_pos[5]], mode='markers', marker=dict(size=10, color='#ffffff', symbol='diamond', line=dict(color='#009E73', width=3)), name='Optimal P2'), row=1, col=1)

    fig.add_trace(go.Scatter3d(
        x=grid_points[:, 0], y=grid_points[:, 1], z=grid_points[:, 2], mode='markers',
        marker=dict(size=5, color=log_fitness_grid, colorscale='RdYlGn_r', cmin=cmin_val, cmax=cmax_val, showscale=True, opacity=0.35, colorbar=dict(title='Log Cost', x=1.02, thickness=15)),
        text=[f"Cost: {f:,.0f}" for f in frozen_fitness_grid], hoverinfo='text', name='City Evaluation'
    ), row=1, col=2)
    fig.add_trace(go.Scatter3d(x=[gbest_pos[3]], y=[gbest_pos[4]], z=[gbest_pos[5]], mode='markers', marker=dict(size=12, color='#ffffff', symbol='diamond', line=dict(color='#009E73', width=3)), name='Optimal P2'), row=1, col=2)

    fig.update_layout(
        template='plotly_white', paper_bgcolor='#ffffff', plot_bgcolor='#ffffff', title="3D Fitness Landscape Analysis (P1 & P3 Frozen)", height=700, margin=dict(l=10, r=10, t=60, b=10),
        scene=dict(xaxis=dict(range=[0, SPACE_X], title='X (m)', backgroundcolor='#ffffff', gridcolor='#e5e5e5'), yaxis=dict(range=[0, SPACE_Y], title='Y (m)', backgroundcolor='#ffffff', gridcolor='#e5e5e5'), zaxis=dict(range=[0, SPACE_Z], title='Z (m)', backgroundcolor='#ffffff', gridcolor='#e5e5e5'), aspectmode='manual', aspectratio=dict(x=1, y=1, z=0.4), camera=dict(eye=dict(x=-1.5, y=-1.5, z=0.8))),
        scene2=dict(xaxis=dict(range=[0, SPACE_X], title='X (m)', backgroundcolor='#ffffff', gridcolor='#e5e5e5'), yaxis=dict(range=[0, SPACE_Y], title='Y (m)', backgroundcolor='#ffffff', gridcolor='#e5e5e5'), zaxis=dict(range=[0, SPACE_Z], title='Z (m)', backgroundcolor='#ffffff', gridcolor='#e5e5e5'), aspectmode='manual', aspectratio=dict(x=1, y=1, z=0.4), camera=dict(eye=dict(x=-1.5, y=-1.5, z=0.8))),
        showlegend=False
    )
    
    fig.show()

def animate_swarm_convergence(swarm_p2_history, swarm_iterations, gbest_pos):
    P1_opt = gbest_pos[0:3]
    P3_opt = gbest_pos[6:9]

    frozen_fitness_swarm = []
    for p2 in swarm_p2_history:
        temp_particle = np.concatenate([P1_opt, p2, P3_opt])
        fit_val, _ = fitness(temp_particle)
        frozen_fitness_swarm.append(fit_val)

    frozen_fitness_swarm = np.array(frozen_fitness_swarm)
    log_fitness_swarm = np.log10(frozen_fitness_swarm + 1)

    cmin_val = log_fitness_swarm.min()
    cmax_val = log_fitness_swarm.max()

    unique_iters = np.unique(swarm_iterations)
    
    num_obstacles = len(OBSTACLES)
    swarm_trace_index = num_obstacles  

    frames = []
    for it in unique_iters:
        idx = (swarm_iterations == it)
        x_iter = swarm_p2_history[idx, 0]
        y_iter = swarm_p2_history[idx, 1]
        z_iter = swarm_p2_history[idx, 2]
        color_iter = log_fitness_swarm[idx]
        cost_iter = frozen_fitness_swarm[idx]

        frames.append(go.Frame(
            data=[go.Scatter3d(
                x=x_iter, y=y_iter, z=z_iter,
                mode='markers',
                marker=dict(
                    size=6,
                    color=color_iter,
                    colorscale='RdYlGn_r',
                    cmin=cmin_val, cmax=cmax_val,
                    opacity=0.9
                ),
                text=[f"Iter: {it}<br>Cost: {c:,.0f}" for c in cost_iter],
                hoverinfo='text'
            )],
            name=str(it),
            traces=[swarm_trace_index] 
        ))

    fig = go.Figure()

    for obs in OBSTACLES:
        x0, x1, y0, y1, z0, z1 = obs
        x_box = [x0, x0, x1, x1, x0, x0, x1, x1]
        y_box = [y0, y1, y1, y0, y0, y1, y1, y0]
        z_box = [z0, z0, z0, z0, z1, z1, z1, z1]
        fig.add_trace(go.Mesh3d(
            x=x_box, y=y_box, z=z_box, 
            alphahull=0, color='#56B4E9', opacity=0.15, hoverinfo='skip'
        ))

    init_idx = (swarm_iterations == unique_iters[0])
    fig.add_trace(go.Scatter3d(
        x=swarm_p2_history[init_idx, 0],
        y=swarm_p2_history[init_idx, 1],
        z=swarm_p2_history[init_idx, 2],
        mode='markers',
        marker=dict(
            size=6,
            color=log_fitness_swarm[init_idx],
            colorscale='RdYlGn_r',
            cmin=cmin_val, cmax=cmax_val,
            showscale=True,
            colorbar=dict(title='Log Cost', x=1.02, thickness=15),
            opacity=0.9
        ),
        text=[f"Iter: {unique_iters[0]}<br>Cost: {c:,.0f}" for c in frozen_fitness_swarm[init_idx]],
        hoverinfo='text',
        name='Swarm'
    ))

    fig.add_trace(go.Scatter3d(
        x=[gbest_pos[3]], y=[gbest_pos[4]], z=[gbest_pos[5]],
        mode='markers',
        marker=dict(size=12, color='#ffffff', symbol='diamond', line=dict(color='#009E73', width=3)),
        name='Optimal P2'
    ))

    fig.update_layout(
        template='plotly_white',
        title="MSIPSO Swarm Convergence Animation",
        height=800,
        margin=dict(l=10, r=10, t=60, b=10),
        scene=dict(
            xaxis=dict(range=[0, SPACE_X], title='X (m)', backgroundcolor='#ffffff', gridcolor='#e5e5e5'),
            yaxis=dict(range=[0, SPACE_Y], title='Y (m)', backgroundcolor='#ffffff', gridcolor='#e5e5e5'),
            zaxis=dict(range=[0, SPACE_Z], title='Z (m)', backgroundcolor='#ffffff', gridcolor='#e5e5e5'),
            aspectmode='manual', aspectratio=dict(x=1, y=1, z=0.4),
            camera=dict(eye=dict(x=-1.5, y=-1.5, z=0.8))
        ),
        showlegend=False,
        updatemenus=[dict(
            type='buttons',
            showactive=False,
            y=0.1, x=0.0,
            xanchor='right', yanchor='top',
            buttons=[
                dict(label='Play',
                     method='animate',
                     args=[None, dict(frame=dict(duration=200, redraw=True), transition=dict(duration=0), fromcurrent=True, mode='immediate')]),
                dict(label='Pause',
                     method='animate',
                     args=[[None], dict(frame=dict(duration=0, redraw=False), mode='immediate')])
            ]
        )],
        sliders=[dict(
            active=0,
            y=0.0, x=0.1,
            len=0.9,
            xanchor='left', yanchor='top',
            steps=[dict(method='animate', args=[[str(it)], dict(mode='immediate', frame=dict(duration=200, redraw=True), transition=dict(duration=0))], label=str(it)) for it in unique_iters]
        )]
    )

    fig.frames = frames
    fig.show()

In [17]:
# --- 5: Validation ---
def verify_all(gbest_pos):
    ctrl = np.vstack([START, gbest_pos[0:3], gbest_pos[3:6], gbest_pos[6:9], TARGET])
    path = generate_path(ctrl)
    labels = ['S', 'P1', 'P2', 'P3', 'T']
    expect = [True, False, False, False, True]
    #print("\n  ── Fix 1: B-Spline Approximation ──────────────────────────")
    all_ok = True
    for cp, lbl, should_be_on in zip(ctrl, labels, expect):
        min_d   = np.linalg.norm(path - cp, axis=1).min()
        on_curve = min_d < 1.0
        ok = (on_curve == should_be_on)
        if not ok: all_ok = False
        #print(f"    {lbl:2s}  dist={min_d:8.3f}  {'ON ' if on_curve else 'OFF'}  [{'OK' if ok else 'FAIL'}]")
    #print(f"    → {'PASS' if all_ok else 'FAIL'}")

    rng = np.random.default_rng(0)
    r4  = rng.random((1000, 9))
    old_z = (r4 * (P_MAX - P_MIN))[:, 2::3].ravel()
    new_z = (P_MIN + r4 * (P_MAX - P_MIN))[:, 2::3].ravel()
    #print("\n  ── Fix 2: D3 Altitude Bounds ──────────────────────────────")
    #print(f"    OLD z ∈ [{old_z.min():.1f}, {old_z.max():.1f}]   violations: {(old_z < H_MIN).sum()}/3000")
    #print(f"    NEW z ∈ [{new_z.min():.1f}, {new_z.max():.1f}]  violations: {(new_z < H_MIN).sum()}/3000")

    #print("\n  ── Fix 3: Path Clearance ───────────────────────────────────")
    collisions = count_collisions(path)
    z_vals = path[:, 2]
    #print(f"    Collisions  : {collisions}")
    #print(f"    Path Z range: [{z_vals.min():.1f}, {z_vals.max():.1f}] m")
    
    clear = True
    for i, pt in enumerate(path):
        for j, obs in enumerate(OBSTACLES):
            if obs[0] <= pt[0] <= obs[1] and obs[2] <= pt[1] <= obs[3]:
                above = pt[2] > obs[5]
                if not above: clear = False
    #if clear:
        #print("    All obstacle overflights confirmed ABOVE roof → PASS\n")

def print_summary(gbest_pos, gbest_fit, best_path, history):
    P1, P2, P3 = gbest_pos[0:3], gbest_pos[3:6], gbest_pos[6:9]
    total_len = np.linalg.norm(np.diff(best_path, axis=0), axis=1).sum()
    c         = count_collisions(best_path)
    z         = best_path[:, 2]
    alt_ok    = bool(np.all((z >= H_MIN) & (z < H_MAX)))
    print("═" * 62)
    print("  MSIPSO  –  RESULTS SUMMARY")
    print("═" * 62)
    print(f"  Best fitness    : {gbest_fit:.4f}")
    print(f"  Path length     : {total_len:.2f} m")
    print(f"  Collisions      : {c}")
    print(f"  Altitude valid  : {alt_ok}  (z ∈ [{z.min():.1f}, {z.max():.1f}] m)")
    print(f"  Control pt P1   : ({P1[0]:.1f}, {P1[1]:.1f}, {P1[2]:.1f})")
    print(f"  Control pt P2   : ({P2[0]:.1f}, {P2[1]:.1f}, {P2[2]:.1f})")
    print(f"  Control pt P3   : ({P3[0]:.1f}, {P3[1]:.1f}, {P3[2]:.1f})")
    print(f"  Best iter       : {int(np.argmin(history))+1}  → fitness {min(history):.4f}")
    print("═" * 62)



In [ ]:
# --- 5: Execution ---
if __name__ == "__main__":
    print("═" * 62)
    print("  MSIPSO – UAV PATH PLANNING  (Interactive Plotly)")
    print(f"  Particles={N_PARTICLES}   Iterations={T_MAX}")
    print(f"  F_thresh={F_THRESH:.2f}")
    print("═" * 62 + "\n")

    gbest_pos, gbest_fit, best_path, history, swarm_p2_history, swarm_iterations = run_msipso(seed=42, verbose=True)
    
    verify_all(gbest_pos)
    print_summary(gbest_pos, gbest_fit, best_path, history)
    
    plot_results_plotly(best_path, history, gbest_pos)
    plot_combined_heatmaps(swarm_p2_history, swarm_iterations, gbest_pos)
    animate_swarm_convergence(swarm_p2_history, swarm_iterations, gbest_pos)

══════════════════════════════════════════════════════════════
  MSIPSO – UAV PATH PLANNING  (Interactive Plotly)
  Particles=50   Iterations=100
  F_thresh=9225101.63
══════════════════════════════════════════════════════════════

  Iter   1/100  [optimising]  fitness=  22251.6192  w=0.900  c1=2.466  c2=0.534
  Iter  10/100  [optimising]  fitness=  19515.2819  w=0.832  c1=2.183  c2=0.817
  Iter  20/100  [optimising]  fitness=  18929.8512  w=0.762  c1=1.909  c2=1.091
  Iter  30/100  [optimising]  fitness=  18680.8341  w=0.700  c1=1.669  c2=1.331
  Iter  40/100  [optimising]  fitness=  18651.3974  w=0.643  c1=1.454  c2=1.546
  Iter  50/100  [optimising]  fitness=  18636.4506  w=0.592  c1=1.260  c2=1.740
  Iter  60/100  [optimising]  fitness=  18632.5608  w=0.545  c1=1.083  c2=1.917
  Iter  70/100  [optimising]  fitness=  18631.3381  w=0.503  c1=0.921  c2=2.079
  Iter  80/100  [optimising]  fitness=  18629.7980  w=0.465  c1=0.770  c2=2.230
  Iter  90/100  [optimising]  fitness=  18627.69